In [5]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# =====================================
# 1. 모델 정의 (제공해주신 구조)
# =====================================
class StaticConditionedEarlyExitGRU(nn.Module):
    def __init__(self, input_dim, static_dim, hidden_dim=128, exit_threshold=0.9):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.exit_threshold = exit_threshold

        self.static_encoder = nn.Sequential(
            nn.Linear(static_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.h0_proj = nn.Linear(hidden_dim, hidden_dim)
        self.gate_proj = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
        self.gru_cell = nn.GRUCell(input_dim, hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        self.exit_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x, static_features, lengths):
        device = x.device
        B, T, F = x.shape
        z = self.static_encoder(static_features)
        h = self.h0_proj(z)
        gate = self.gate_proj(z)

        final_logits = torch.zeros(B, 1, device=device)
        exit_steps = torch.zeros(B, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        
        all_logits = []
        all_exit_probs = []

        for t in range(T):
            x_t = x[:, t, :]
            gated_x_t = x_t * gate
            h = self.gru_cell(gated_x_t, h)
            
            logits = self.classifier(h)
            exit_prob = self.exit_head(h)
            
            all_logits.append(logits)
            all_exit_probs.append(exit_prob)

            # 조기 종료 로직 (추론 시 사용)
            should_exit = (exit_prob.squeeze(-1) > self.exit_threshold) & (~finished) & (t < lengths)
            final_logits[should_exit] = logits[should_exit]
            exit_steps[should_exit] = t
            finished = finished | should_exit

        # 끝까지 종료 안 된 샘플 처리
        never_finished = ~finished
        if never_finished.any():
            final_logits[never_finished] = logits[never_finished]
            exit_steps[never_finished] = lengths[never_finished] - 1

        return {
            "logits": final_logits, 
            "all_logits": all_logits, 
            "exit_steps": exit_steps
        }

# =====================================
# 2. 데이터셋 및 전처리 (Direction 인코딩 포함)
# =====================================
class FlowDataset(Dataset):
    def __init__(self, file_path):
        self.samples = []
        with open(file_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))
        
        # Direction 매핑
        self.dir_map = {"src_to_dst": 0.0, "dst_to_src": 1.0}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        x = torch.tensor(item['x'], dtype=torch.float32) # [T, 11]
        
        # direction만 static feature로 사용
        direction_str = item['flow_key']['direction']
        static_val = self.dir_map.get(direction_str, 0.0)
        static_f = torch.tensor([static_val], dtype=torch.float32)
        
        label = torch.tensor([item['label']], dtype=torch.float32)
        length = torch.tensor(len(item['x']), dtype=torch.long)
        
        return x, static_f, length, label

def collate_fn(batch):
    """길이가 다른 시퀀스를 패딩하여 배치로 만듭니다."""
    xs, statics, lengths, labels = zip(*batch)
    xs_padded = pad_sequence(xs, batch_first=True) # [B, T_max, F]
    statics = torch.stack(statics)
    lengths = torch.stack(lengths)
    labels = torch.stack(labels)
    return xs_padded, statics, lengths, labels

# =====================================
# 3. 학습 및 저장 메인 루프
# =====================================
def train_model():
    # 설정
    INPUT_DIM = 11   # x의 피처 개수
    STATIC_DIM = 1  # direction 하나
    HIDDEN_DIM = 64
    BATCH_SIZE = 16
    EPOCHS = 20
    LEARNING_RATE = 0.001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 데이터 준비
    dataset = FlowDataset('dataset.jsonl')
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

    # 모델 및 최적화
    model = StaticConditionedEarlyExitGRU(INPUT_DIM, STATIC_DIM, HIDDEN_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()

    print(f"Training started on {DEVICE}...")

    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0
        for batch_x, batch_static, batch_len, batch_label in dataloader:
            batch_x, batch_static, batch_len, batch_label = \
                batch_x.to(DEVICE), batch_static.to(DEVICE), batch_len.to(DEVICE), batch_label.to(DEVICE)

            optimizer.zero_grad()
            
            outputs = model(batch_x, batch_static, batch_len)
            
            # Loss 계산: 최종 출력뿐만 아니라 중간 모든 단계의 logit에 대해 loss를 계산하여
            # 어느 시점에서 exit하더라도 정확하도록 유도합니다 (Joint Optimization).
            main_loss = criterion(outputs['logits'], batch_label)
            
            step_loss = 0
            for step_logit in outputs['all_logits']:
                step_loss += criterion(step_logit, batch_label)
            
            total_loss = main_loss + (step_loss / len(outputs['all_logits']))
            
            total_loss.backward()
            optimizer.step()
            epoch_loss += total_loss.item()

        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {epoch_loss/len(dataloader):.4f}")

    # 가중치 저장
    torch.save(model.state_dict(), "flow_model_weights.pth")
    print("Model weights saved to 'flow_model_weights.pth'")

if __name__ == "__main__":
    # dataset.jsonl 파일이 같은 경로에 있어야 실행됩니다.
    train_model()

Training started on cpu...
Epoch [1/20], Loss: 0.6486
Epoch [2/20], Loss: 0.5152
Epoch [3/20], Loss: 0.4937
Epoch [4/20], Loss: 0.4918
Epoch [5/20], Loss: 0.4965
Epoch [6/20], Loss: 0.4900
Epoch [7/20], Loss: 0.4949
Epoch [8/20], Loss: 0.4958
Epoch [9/20], Loss: 0.4932
Epoch [10/20], Loss: 0.4915


KeyboardInterrupt: 

In [6]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

# [모델 클래스는 이전과 동일하므로 생략하거나 그대로 사용하세요]
# ... (StaticConditionedEarlyExitGRU 정의 부분)

class FlowDataset(Dataset):
    def __init__(self, file_path):
        self.samples = []
        with open(file_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))
        self.dir_map = {"src_to_dst": 0.0, "dst_to_src": 1.0}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        x = torch.tensor(item['x'], dtype=torch.float32)
        direction_str = item['flow_key']['direction']
        static_val = self.dir_map.get(direction_str, 0.0)
        static_f = torch.tensor([static_val], dtype=torch.float32)
        label = torch.tensor([item['label']], dtype=torch.float32)
        length = torch.tensor(len(item['x']), dtype=torch.long)
        return x, static_f, length, label

def collate_fn(batch):
    xs, statics, lengths, labels = zip(*batch)
    xs_padded = pad_sequence(xs, batch_first=True)
    statics = torch.stack(statics)
    lengths = torch.stack(lengths)
    labels = torch.stack(labels)
    return xs_padded, statics, lengths, labels

def train_and_validate():
    # 하이퍼파라미터
    INPUT_DIM = 11
    STATIC_DIM = 1
    HIDDEN_DIM = 64
    BATCH_SIZE = 16
    EPOCHS = 30
    LR = 0.001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. 데이터 로드 및 분할
    full_dataset = FlowDataset('dataset.jsonl')
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    # 2. 모델 및 도구 설정
    model = StaticConditionedEarlyExitGRU(INPUT_DIM, STATIC_DIM, HIDDEN_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float('inf')

    for epoch in range(EPOCHS):
        # --- Training Phase ---
        model.train()
        train_loss = 0
        for b_x, b_s, b_len, b_y in train_loader:
            b_x, b_s, b_len, b_y = b_x.to(DEVICE), b_s.to(DEVICE), b_len.to(DEVICE), b_y.to(DEVICE)
            
            optimizer.zero_grad()
            out = model(b_x, b_s, b_len)
            
            # Loss: Final Output + Mean of All Steps
            loss = criterion(out['logits'], b_y)
            step_loss = sum(criterion(sl, b_y) for sl in out['all_logits']) / len(out['all_logits'])
            total_loss = loss + step_loss
            
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()

        # --- Validation Phase ---
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for b_x, b_s, b_len, b_y in val_loader:
                b_x, b_s, b_len, b_y = b_x.to(DEVICE), b_s.to(DEVICE), b_len.to(DEVICE), b_y.to(DEVICE)
                
                out = model(b_x, b_s, b_len)
                v_loss = criterion(out['logits'], b_y)
                val_loss += v_loss.item()
                
                # 정확도 계산 (0.5 임계값 기준)
                preds = torch.sigmoid(out['logits']) > 0.5
                correct += (preds == b_y).sum().item()
                total += b_y.size(0)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        accuracy = (correct / total) * 100

        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {accuracy:.2f}%")

        # 3. 최적의 모델 저장 (Best Model 체크포인트)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_flow_model.pth")
            print(f"--> Best model saved with Val Loss: {best_val_loss:.4f}")

if __name__ == "__main__":
    train_and_validate()

Epoch [1/30] Train Loss: 0.7476 | Val Loss: 0.2668 | Val Acc: 91.01%
--> Best model saved with Val Loss: 0.2668
Epoch [2/30] Train Loss: 0.5266 | Val Loss: 0.2390 | Val Acc: 91.01%
--> Best model saved with Val Loss: 0.2390
Epoch [3/30] Train Loss: 0.5049 | Val Loss: 0.2351 | Val Acc: 91.01%
--> Best model saved with Val Loss: 0.2351
Epoch [4/30] Train Loss: 0.5006 | Val Loss: 0.2301 | Val Acc: 91.01%
--> Best model saved with Val Loss: 0.2301
Epoch [5/30] Train Loss: 0.4945 | Val Loss: 0.2361 | Val Acc: 91.01%
Epoch [6/30] Train Loss: 0.5007 | Val Loss: 0.2327 | Val Acc: 91.01%
Epoch [7/30] Train Loss: 0.4970 | Val Loss: 0.2347 | Val Acc: 91.01%
Epoch [8/30] Train Loss: 0.4982 | Val Loss: 0.2336 | Val Acc: 91.01%
Epoch [9/30] Train Loss: 0.4942 | Val Loss: 0.2296 | Val Acc: 91.01%
--> Best model saved with Val Loss: 0.2296
Epoch [10/30] Train Loss: 0.5041 | Val Loss: 0.2360 | Val Acc: 91.01%
Epoch [11/30] Train Loss: 0.5048 | Val Loss: 0.2356 | Val Acc: 91.01%
Epoch [12/30] Train Loss

In [7]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

# =====================================
# 1. 모델 정의 (제공해주신 구조)
# =====================================
class Basic1DCNN(nn.Module):
    def __init__(self, num_features=11):
        super(Basic1DCNN, self).__init__()
        
        # [Batch, 11, Time] 형태로 입력받음 (forward에서 permute함)
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        
        # 고정된 길이 10을 입력받았을 때 결과가 4가 되어 32*4가 됨
        self.fc = nn.Linear(in_features=32 * 4, out_features=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [Batch, Time, Features] -> [Batch, Features, Time]
        x = x.permute(0, 2, 1) 

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.flatten(x)
        
        x = self.fc(x)
        out = self.sigmoid(x) 
        
        return out

# =====================================
# 2. 데이터셋 정의 (시계열 x만 사용)
# =====================================
class CNNDataset(Dataset):
    def __init__(self, file_path, seq_len=10):
        self.samples = []
        self.seq_len = seq_len
        with open(file_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        x = torch.tensor(item['x'], dtype=torch.float32)
        
        # 모델의 Linear(32*4)에 맞추기 위해 길이를 고정 (10개 패킷)
        if x.size(0) > self.seq_len:
            x = x[:self.seq_len, :] # 자르기
        elif x.size(0) < self.seq_len:
            # 부족하면 0으로 채우기 (Padding)
            padding = torch.zeros((self.seq_len - x.size(0)), x.size(1))
            x = torch.cat([x, padding], dim=0)
            
        label = torch.tensor([item['label']], dtype=torch.float32)
        return x, label

# =====================================
# 3. 학습 및 검증 통합 함수
# =====================================
def train_cnn_model():
    # 하이퍼파라미터
    INPUT_DIM = 11
    SEQ_LEN = 10  # CNN 구조상 고정된 길이
    BATCH_SIZE = 16
    EPOCHS = 30
    LR = 0.001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 데이터 로드
    dataset = CNNDataset('dataset.jsonl', seq_len=SEQ_LEN)
    
    # 데이터 분할 (8:2)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 모델, 손실함수, 최적화
    model = Basic1DCNN(num_features=INPUT_DIM).to(DEVICE)
    # 모델 내부에 Sigmoid가 있으므로 BCELoss를 사용합니다.
    criterion = nn.BCELoss() 
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_val_loss = float('inf')

    print(f"CNN Training started on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- 학습 ---
        model.train()
        train_loss = 0
        for b_x, b_y in train_loader:
            b_x, b_y = b_x.to(DEVICE), b_y.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(b_x)
            loss = criterion(outputs, b_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()

        # --- 검증 ---
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for b_x, b_y in val_loader:
                b_x, b_y = b_x.to(DEVICE), b_y.to(DEVICE)
                outputs = model(b_x)
                
                v_loss = criterion(outputs, b_y)
                val_loss += v_loss.item()
                
                preds = (outputs > 0.5).float()
                correct += (preds == b_y).sum().item()
                total += b_y.size(0)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        acc = (correct / total) * 100

        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {acc:.2f}%")

        # 최적 모델 저장
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_cnn_flow_model.pth")
            print(f"--> Best model saved!")

if __name__ == "__main__":
    train_cnn_model()

CNN Training started on cpu...
Epoch [1/30] Train Loss: 9.7321 | Val Loss: 10.9954 | Val Acc: 89.21%
--> Best model saved!
Epoch [2/30] Train Loss: 9.6429 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [3/30] Train Loss: 9.6429 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [4/30] Train Loss: 9.5536 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [5/30] Train Loss: 9.4643 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [6/30] Train Loss: 9.4643 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [7/30] Train Loss: 9.7321 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [8/30] Train Loss: 9.5536 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [9/30] Train Loss: 9.5536 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [10/30] Train Loss: 9.4643 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [11/30] Train Loss: 9.5536 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [12/30] Train Loss: 9.5536 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [13/30] Train Loss: 9.4643 | Val Loss: 10.9954 | Val Acc: 89.21%
Epoch [14/30] Train Loss: 9.5536 